# LangGraph & LangSmith Terminology — Explained in Plain Words

### Every term pointed at a real line of code in one small case study

---

## How this notebook works

Most glossaries define a word using three other words you also don't know. This one does something different.

We build **one small working assistant** first (§1). Then, for every piece of jargon, you get the same four things:



## The case study: ShopBot

> A tiny support assistant for an online shop. A customer writes in. ShopBot works out what they want, looks up their order, drafts a reply, checks its own draft, and — if money is involved — **stops and waits for a human** to approve.

It's deliberately small (about 60 lines) but it contains **every concept** in both frameworks: branching, looping, memory, pausing, and tracing.

## If you've already built CineGuide or MeetingMind

ShopBot is deliberately tiny — six steps, a fake order database, no vector store — so that **nothing distracts from the vocabulary**. Every term maps straight onto the bigger projects:

| Term | In ShopBot (here) | In CineGuide (Netflix) | In MeetingMind (Chroma) |
|---|---|---|---|
| State fields | messages, intent, order, refund | messages, route, candidates, stats | messages, route, owner, context |
| The router's choices | track / refund / question | recommend / stats / chitchat | recall / owner / chitchat |
| Nodes | 6 | 7 | 5 |
| `thread_id` | one customer | one user session | one user session |
| Dataset | 6 test messages | 12 catalogue questions | 12 meeting questions |
| Interrupt | refunds over a threshold | (not used) | (not used) |

So once these 34 words make sense here, re-reading either bigger notebook should feel like reading a sentence rather than decoding one.

## Contents

| § | Topic |
|---|---|
| 0 | Setup |
| 1 | **The case study** — build ShopBot |
| 2 | **LangGraph terminology** — 18 terms |
| 3 | **LangSmith terminology** — 14 terms |
| 4 | Two cheat sheets + words you'll hear but don't need yet |

## 0. Setup

 No version numbers in the install — pinning packages to guessed versions makes pip fail, and pip is all-or-nothing, so *nothing* installs.

In [ ]:
%pip install -q -U langchain langchain-openai langgraph langsmith

print("Install done. If a 'RESTART SESSION' button appeared, click it and continue.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 14.3 MB/s eta 0:00:00
Install done. If a 'RESTART SESSION' button appeared, click it and continue.


In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (sk-...): ")

MODEL = "gpt-4o-mini"

try:
    from langchain_openai import ChatOpenAI
    from langgraph.graph import StateGraph
    from langsmith import traceable
except ImportError as e:
    print("PROBLEM:", e, "\n-> re-run install, RESTART session, run this again.")
else:
    llm = ChatOpenAI(model=MODEL, temperature=0)
    print("test ->", llm.invoke("Reply with one word: ready").content)
    print("ALL GOOD.")


Paste your OpenAI API key (sk-...): ··········
test -> Ready.
ALL GOOD.


In [ ]:
# Everything we need, in one place.
from typing import TypedDict, Annotated, Literal, Optional, List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from pydantic import BaseModel, Field
from langsmith import traceable
import time, json

print("imports ready")

imports ready


# 1. The case study — ShopBot

Read this section quickly. You don't need to understand every word yet — that's what §2 and §3 are for. This is just so the terminology has something real to point at.

### What ShopBot does

```
        customer writes in
                │
          ┌─────────────┐
          │   classify  │   what do they want?
          └──┬───┬───┬──┘
             │   │   └──────────────► chat ──────────┐
             │   └──► look_up_order ──► draft ──┐     │
             │                          ▲       │     │
             │                          └─ fix ─┤     │  (up to 2 tries)
             │                                  │     │
             └──► look_up_order ──► draft ──► check    │
                                              │       │
                                    money? ──► wait_for_human ⏸
                                              │       │
                                            send ─────┘
                                              │
                                             done
```

In [ ]:
# ---------- our pretend order database ----------
ORDERS = {
    "A-100": {"item": "desk lamp",  "status": "DELIVERED", "days": 3,  "price": 42.00},
    "B-200": {"item": "monitor",    "status": "DELAYED",   "days": 11, "price": 310.00},
}

def look_up(order_id: str) -> str:
    """An ordinary Python function. No AI involved."""
    o = ORDERS.get((order_id or "").strip().upper())
    if not o:
        return f"No order found with id {order_id}."
    return (f"Order {order_id}: {o['item']}, status {o['status']}, "
            f"ordered {o['days']} days ago, price ${o['price']:.2f}.")

def price_of(order_id: str) -> float:
    """How much money is at stake. Returns 0.0 if we don't know the order."""
    return ORDERS.get((order_id or "").upper(), {}).get("price", 0.0)

print(look_up("b-200"))
print(look_up("Z-999"))

Order b-200: monitor, status DELAYED, ordered 11 days ago, price $310.00.
No order found with id Z-999.


In [ ]:
# ---------- what ShopBot keeps track of ----------
class ShopState(TypedDict):
    messages: Annotated[list, add_messages]   # the conversation
    intent: str                                # "track" / "refund" / "chat"
    order_id: Optional[str]
    order_info: str
    draft: str
    critique: str
    attempts: int                              # how many times we've tried drafting
    approved: Optional[bool]                   # did a human approve the refund?

# ---------- what the model must decide, in a fixed shape ----------
class Intent(BaseModel):
    """What does the customer want?"""
    intent: Literal["track", "refund", "chat"] = Field(
        description="'track' = where is my order. 'refund' = wants money back. 'chat' = greeting or thanks.")
    order_id: Optional[str] = Field(default=None, description="Order id like 'A-100', else null.")

class Check(BaseModel):
    """Is the drafted reply good enough to send?"""
    ok: bool = Field(description="True only if the draft states the real status and a next step.")
    problem: str = Field(description="What to fix, or 'none'.")

classifier = llm.with_structured_output(Intent)
checker = llm.with_structured_output(Check)

print("state and schemas ready")

state and schemas ready


In [ ]:
# ---------- the six steps ----------
def classify(state: ShopState) -> dict:
    r = classifier.invoke(f"Classify this customer message:\n{state['messages'][-1].content}")
    print(f"   classify -> {r.intent} (order {r.order_id})")
    return {"intent": r.intent, "order_id": r.order_id, "attempts": 0}

def look_up_order(state: ShopState) -> dict:
    info = look_up(state["order_id"]) if state.get("order_id") else "No order id given."
    print("   look_up_order -> done")
    return {"order_info": info}

def draft(state: ShopState) -> dict:
    fix = f"\nA reviewer rejected your last try. Fix this: {state['critique']}" if state.get("critique") else ""
    out = llm.invoke(
        f"You are a shop support agent. Write a reply under 60 words.\n"
        f"Use only these facts: {state['order_info']}\n"
        f"Customer said: {state['messages'][-1].content}{fix}"
    )
    n = state.get("attempts", 0) + 1
    print(f"   draft -> try {n}")
    return {"draft": out.content, "attempts": n}

def check(state: ShopState) -> dict:
    r = checker.invoke(f"FACTS: {state['order_info']}\n\nDRAFT: {state['draft']}")
    print(f"   check -> {'ok' if r.ok else 'rejected'}")
    return {"critique": "" if r.ok else r.problem, "approved": state.get("approved")}

def wait_for_human(state: ShopState) -> dict:
    """Stops the whole thing and waits for a person."""
    money = price_of(state.get("order_id"))
    print(f"   wait_for_human -> PAUSED (${money:.2f})")
    decision = interrupt({"question": f"Approve a ${money:.2f} refund?", "draft": state["draft"]})
    return {"approved": bool(decision.get("approve"))}

def send(state: ShopState) -> dict:
    text = state["draft"]
    if state.get("intent") == "refund":
        text += ("\n\nYour refund is approved." if state.get("approved")
                 else "\n\nA specialist will review your refund request.")
    print("   send -> done")
    return {"messages": [AIMessage(text)]}

def chat(state: ShopState) -> dict:
    out = llm.invoke([SystemMessage("You are ShopBot. One friendly sentence."),
                      *state["messages"][-3:]])
    print("   chat -> done")
    return {"messages": [out]}

print("6 steps defined")

6 steps defined


In [ ]:
# ---------- wire the steps together ----------
def after_classify(state: ShopState) -> str:
    return "chat" if state["intent"] == "chat" else "look_up_order"

def after_check(state: ShopState) -> str:
    if state["critique"] and state["attempts"] < 2:   # not good enough, and tries left
        return "draft"                                 # go BACK and try again
    if state["intent"] == "refund" and price_of(state.get("order_id")) > 50:
        return "wait_for_human"                        # money -> ask a person
    return "send"

builder = StateGraph(ShopState)
for name, fn in [("classify", classify), ("look_up_order", look_up_order), ("draft", draft),
                 ("check", check), ("wait_for_human", wait_for_human), ("send", send),
                 ("chat", chat)]:
    builder.add_node(name, fn)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", after_classify, ["look_up_order", "chat"])
builder.add_edge("look_up_order", "draft")
builder.add_edge("draft", "check")
builder.add_conditional_edges("check", after_check, ["draft", "wait_for_human", "send"])
builder.add_edge("wait_for_human", "send")
builder.add_edge("send", END)
builder.add_edge("chat", END)

shopbot = builder.compile(checkpointer=InMemorySaver())

print("ShopBot is built. Nodes:",
      [n for n in shopbot.get_graph().nodes if not n.startswith("__")])

ShopBot is built. Nodes: ['classify', 'look_up_order', 'draft', 'check', 'wait_for_human', 'send', 'chat']


In [ ]:
# ---------- run it once, so we have something to talk about ----------
cfg = {"configurable": {"thread_id": "customer-1"}}

print("CUSTOMER: Where is my order B-200?\n")
result = shopbot.invoke({"messages": [HumanMessage("Where is my order B-200?")],
                         "attempts": 0}, cfg)
print("\nSHOPBOT:", result["messages"][-1].content)

CUSTOMER: Where is my order B-200?

   classify -> track (order B-200)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done

SHOPBOT: Dear Customer,  

Your order B-200 for the monitor, priced at $310.00, is currently delayed. It was placed 11 days ago. We apologize for the inconvenience and are working to resolve the issue. Thank you for your patience.  

Best regards,  
[Your Name]  


# 2. LangGraph terminology

Eighteen terms. Each one points at a line you just ran.

---

## 2.1 Graph

| | |
|---|---|
| 🗣️ **Plain words** | Your app drawn as a **flowchart**: boxes (steps) joined by arrows. |
| 🏠 **Analogy** | A London Underground map. Stations and the lines between them. |
| 👉 **In ShopBot** | The whole picture: `classify → look_up_order → draft → check → send` |

The word "graph" is just mathematician-speak for "boxes and arrows". Nothing more.

**Why a graph and not a straight line?** Because ShopBot needs to do things a straight line can't: **branch** (refund vs tracking), **loop back** (redraft a bad reply), and **stop and wait** (human approval).

In [ ]:
# ▶️ SEE IT: print the flowchart. This text is "Mermaid" format, which GitHub draws
# automatically - so you can paste it into a README.
print(shopbot.get_graph().draw_mermaid())

## 2.2 State

| | |
|---|---|
| 🗣️ **Plain words** | One **dictionary** that every step can read from and write to. It carries information through the flowchart. |
| 🏠 **Analogy** | A patient's clipboard in a hospital. Reception writes the name, the nurse adds the blood pressure, the doctor reads it all. Same clipboard, passed along. |
| 👉 **In ShopBot** | `class ShopState(TypedDict):` — the messages, the intent, the order info, the draft |

**`TypedDict`** just means "a dictionary where I've written down which keys exist". At runtime it's an ordinary dict; the type hints are there so you and LangGraph can catch typos.

⚠️ If you try to write a key that isn't in your state class, you get an error. That's the point — it stops silent typos.

In [ ]:
# ▶️ SEE IT: the state after the run. This is the clipboard, fully filled in.
for key, value in result.items():
    shown = f"{len(value)} messages" if key == "messages" else str(value)[:70]
    print(f"  {key:<12}: {shown}")

## 2.3 Node

| | |
|---|---|
| 🗣️ **Plain words** | One **step**. It's just a normal Python function: takes the state, returns what changed. |
| 🏠 **Analogy** | One station on the Underground map. One worker on an assembly line. |
| 👉 **In ShopBot** | `def classify(state)`, `def draft(state)`, `def send(state)` — six of them |

There is **no special class and no decorator**. If it's a function that takes the state and returns a dict, it's a node. That's why nodes are easy to test — they're ordinary functions.

In [ ]:
# ▶️ SEE IT: call a node directly, with no graph and no LLM involved at all.
# This is why nodes being plain functions is such a practical advantage.
fake_state = {"order_id": "A-100", "messages": [], "attempts": 0}
print("look_up_order on its own ->", look_up_order(fake_state))

## 2.4 Partial update (or "merge")

| | |
|---|---|
| 🗣️ **Plain words** | A node returns **only the keys it changed**, not the whole state. LangGraph merges them in. |
| 🏠 **Analogy** | The nurse writes "BP: 120/80" on the clipboard. She doesn't rewrite the patient's name and address. |
| 👉 **In ShopBot** | `return {"order_info": info}` — that node touches one key and leaves the other seven alone |

In [ ]:
# ▶️ SEE IT: look at what each node returned. Notice how small each one is.
# What each node returns - notice how small each one is.
print("classify       returns: intent, order_id, attempts   (3 keys)")
print("look_up_order  returns: order_info                   (1 key)")
print("draft          returns: draft, attempts              (2 keys)")
print("send           returns: messages                     (1 key)")
print("\nour state has 8 keys in total:", len(ShopState.__annotations__))
print("\nAdd them up and you get the full state - but no single node had to know")
print("about all of it. That's what makes it easy to add a step later.")

## 2.5 Edge

| | |
|---|---|
| 🗣️ **Plain words** | An **arrow**: "when this step finishes, do that step next." |
| 🏠 **Analogy** | The track between two stations. There's only one way to go. |
| 👉 **In ShopBot** | `builder.add_edge("look_up_order", "draft")` — always, no decision involved |

## 2.6 START and END

| | |
|---|---|
| 🗣️ **Plain words** | Two built-in markers meaning "begin here" and "stop here". They aren't real steps. |
| 🏠 **Analogy** | The "You are here" arrow and the exit sign. |
| 👉 **In ShopBot** | `builder.add_edge(START, "classify")` and `builder.add_edge("send", END)` |

In [ ]:
# ▶️ SEE IT: START and END show up in the node list with double underscores,
# which is how you can tell they're built-in rather than yours.
print("all nodes including the built-in markers:")
print("  ", list(shopbot.get_graph().nodes))

## 2.7 Conditional edge (also called a "router")

| | |
|---|---|
| 🗣️ **Plain words** | An arrow that **splits**. A small function looks at the state and picks which way to go. |
| 🏠 **Analogy** | A railway points/switch. Or airport security: EU passports left, others right. |
| 👉 **In ShopBot** | `builder.add_conditional_edges("classify", after_classify, ["look_up_order", "chat"])` |

The deciding function returns the **name** of the next node, as a plain string:

```python
def after_classify(state) -> str:
    return "chat" if state["intent"] == "chat" else "look_up_order"
```

⚠️ **Always have a default.** Notice `after_classify` ends with `else` — if `intent` were ever something unexpected, the graph still knows where to go instead of crashing.

In [ ]:
# ▶️ SEE IT: call the routers directly with made-up states. No LLM, instant.
print("after_classify:")
print("   intent=chat   ->", after_classify({"intent": "chat"}))
print("   intent=track  ->", after_classify({"intent": "track"}))

print("\nafter_check:")
print("   bad draft, 1 try used ->",
      after_check({"critique": "too vague", "attempts": 1, "intent": "track"}))
print("   good draft, tracking  ->",
      after_check({"critique": "", "attempts": 1, "intent": "track"}))
print("   good draft, big refund->",
      after_check({"critique": "", "attempts": 1, "intent": "refund", "order_id": "B-200"}))

print("\nThree different destinations from one node. That's a conditional edge.")

## 2.8 Cycle (or "loop")

| | |
|---|---|
| 🗣️ **Plain words** | An arrow that goes **backwards**, so a step can run more than once. |
| 🏠 **Analogy** | Tasting a sauce, adding salt, tasting again. |
| 👉 **In ShopBot** | `after_check` can return `"draft"` — sending us back to a step we already did |

This is the thing a straight-line chain **cannot** do, and it's the main reason LangGraph exists.

## 2.9 Recursion limit (and why you need a counter)

| | |
|---|---|
| 🗣️ **Plain words** | A safety cap on how many steps a run may take, so a loop can't go forever. |
| 🏠 **Analogy** | A kettle that switches itself off. |
| 👉 **In ShopBot** | `state["attempts"] < 2` in `after_check` — our own counter, checked every time |

LangGraph has a built-in limit (default 25 steps) and raises `GraphRecursionError` if you hit it. **But don't rely on it.** Put your own counter in the state, like `attempts`, so *you* decide what happens when the loop gives up.

In [ ]:
# ▶️ SEE IT: the counter is what makes the loop finite.
print("attempts used in our run:", result.get("attempts"))

# Prove the guard works, using plain dicts:
print("\nwhat after_check decides as attempts climb:")
for n in [0, 1, 2, 3]:
    where = after_check({"critique": "still too vague", "attempts": n, "intent": "track"})
    print(f"   attempts={n} -> {where}" + ("   (loops back)" if where == "draft" else "   (gives up and sends)"))

print("\nWithout that counter, a picky reviewer would loop forever - and bill you forever.")

## 2.10 Reducer

| | |
|---|---|
| 🗣️ **Plain words** | The rule for **how a new value combines with the old one**. By default: replace it. |
| 🏠 **Analogy** | Two ways to update a shopping list. **Replace**: throw it away, write a new one. **Add**: append to the bottom. A reducer picks which. |
| 👉 **In ShopBot** | `messages: Annotated[list, add_messages]` — this one means "add to the bottom" |

Read `Annotated[list, add_messages]` as: *"this is a list, and combine updates using `add_messages`."*

**Why it matters:** for a conversation you obviously want to *add* messages, not replace the whole history each turn. Without this, the second message would delete the first.

In [ ]:
# ▶️ SEE IT: the same graph shape with and without a reducer.

class Replace(TypedDict):
    items: list                                        # no reducer -> replaces

class Add(TypedDict):
    items: Annotated[list, lambda old, new: old + new] # a reducer -> adds

def step_a(state): return {"items": ["first"]}
def step_b(state): return {"items": ["second"]}

def tiny(state_class):
    b = StateGraph(state_class)
    b.add_node("a", step_a); b.add_node("b", step_b)
    b.add_edge(START, "a"); b.add_edge("a", "b"); b.add_edge("b", END)
    return b.compile()

print("no reducer (replaces):", tiny(Replace).invoke({"items": []}))
print("with reducer (adds)  :", tiny(Add).invoke({"items": []}))
print("\nSame two steps. Completely different result. That's a reducer.")

## 2.11 `add_messages`

| | |
|---|---|
| 🗣️ **Plain words** | The ready-made reducer for conversations. Adds new messages to the list. |
| 🏠 **Analogy** | Stapling a new page onto a file, rather than shredding the file. |
| 👉 **In ShopBot** | `messages: Annotated[list, add_messages]` |

It does three helpful things:

1. **adds** new messages to the end
2. **converts** shorthand like `{"role": "user", "content": "hi"}` into proper message objects
3. **replaces** a message if the new one has the same `id` (this is how "edit my last message" works)

In [ ]:
# ▶️ SEE IT: use add_messages on its own, outside any graph.
merged = add_messages(
    [HumanMessage("hello")],                            # what we had
    [{"role": "assistant", "content": "hi there"}],     # what's new (shorthand dict)
)
for m in merged:
    print(f"  {m.type:<10} {type(m).__name__:<14} {m.content}")

print("\nNotice the shorthand dict became a real AIMessage object. That's point 2.")

## 2.12 Compile

| | |
|---|---|
| 🗣️ **Plain words** | "I've finished drawing the flowchart — check it and give me something I can run." |
| 🏠 **Analogy** | Handing your architectural drawing to the surveyor. They check the doors connect to rooms, then approve it. |
| 👉 **In ShopBot** | `shopbot = builder.compile(checkpointer=InMemorySaver())` |

Before compiling you have a `builder` (a plan). After compiling you have a **runnable app**. `compile()` catches mistakes like an edge pointing at a node that doesn't exist.

In [ ]:
# ▶️ SEE IT: compile catches a broken flowchart before you ever run it.
bad = StateGraph(ShopState)
bad.add_node("one", lambda s: {})
bad.add_edge(START, "one")
bad.add_edge("one", "a_node_that_does_not_exist")     # ⚠️ a typo

try:
    bad.compile()
    print("compiled (no error)")
except Exception as e:
    print("compile refused it:", type(e).__name__)
    print("  ", str(e)[:130])
    print("\nGood - you found the typo now, not at 2am in production.")

## 2.13 Invoke

| | |
|---|---|
| 🗣️ **Plain words** | "Run it once and give me the answer." |
| 🏠 **Analogy** | Pressing start on a washing machine and waiting for the beep. |
| 👉 **In ShopBot** | `shopbot.invoke({"messages": [...]}, cfg)` |

It's the same word you use for models, prompts and chains in LangChain. Once you know `.invoke()`, most of both frameworks becomes predictable.

## 2.14 Config

| | |
|---|---|
| 🗣️ **Plain words** | The **second** argument to invoke. Settings for *this particular run* — most importantly, which conversation it belongs to. |
| 🏠 **Analogy** | The washing machine's programme dial. Same machine, different settings per load. |
| 👉 **In ShopBot** | `cfg = {"configurable": {"thread_id": "customer-1"}}` |

The nesting (`"configurable"` inside a dict) looks odd but you just copy it. Other things can live in config too, like `recursion_limit`.

In [ ]:
# ▶️ SEE IT: config also holds the safety cap.
print("config we used:", cfg)

# You can override the step limit per run:
strict = {"configurable": {"thread_id": "test-limit"}, "recursion_limit": 5}
print("config with a tighter step limit:", strict)

## 2.15 Stream and `stream_mode`

| | |
|---|---|
| 🗣️ **Plain words** | Instead of waiting for the final answer, get updates **as they happen**. |
| 🏠 **Analogy** | A pizza tracker: "preparing… baking… out for delivery" instead of silence then a doorbell. |
| 👉 **In ShopBot** | `shopbot.stream({...}, cfg, stream_mode="updates")` |

| `stream_mode` | What you get | Good for |
|---|---|---|
| `"updates"` | what each step returned | showing progress: "checking your order…" |
| `"values"` | the whole state after each step | debugging |
| `"messages"` | the model's words as they're typed | the typing effect in a chat window |

In [ ]:
# ▶️ SEE IT: watch the steps go past, one at a time.
print("stream_mode='updates' - one line per step:\n")
for chunk in shopbot.stream(
    {"messages": [HumanMessage("Where is order A-100?")], "attempts": 0},
    {"configurable": {"thread_id": "stream-demo"}},
    stream_mode="updates",
):
    for step_name, what_changed in chunk.items():
        keys = list(what_changed.keys()) if isinstance(what_changed, dict) else "?"
        print(f"   step '{step_name}' changed: {keys}")

print("\nThis is also the best way to SEE the route your run actually took.")

## 2.16 Checkpointer

| | |
|---|---|
| 🗣️ **Plain words** | Something that **saves the state after every step**, so nothing is lost. |
| 🏠 **Analogy** | A video game's autosave. Close the game, come back tomorrow, carry on. |
| 👉 **In ShopBot** | `builder.compile(checkpointer=InMemorySaver())` |

That one argument gives you **four** things at once:

| You get | Because |
|---|---|
| **Memory** | the next message loads the saved conversation |
| **Crash recovery** | the state is saved, not held in one running program |
| **Pausing for a human** | a pause is just a save you haven't continued yet (§2.18) |
| **Separate users** | each conversation is saved under its own name |

| Version | Use for |
|---|---|
| `InMemorySaver` | learning. It's a dictionary — vanishes when you restart |
| `SqliteSaver` | one computer, saved to a file |
| `PostgresSaver` | real apps |

## 2.17 Thread and `thread_id`

| | |
|---|---|
| 🗣️ **Plain words** | A **thread** is one conversation. `thread_id` is its name. |
| 🏠 **Analogy** | A WhatsApp chat. Your chat with Mum and your chat with your boss are separate threads on the same phone. |
| 👉 **In ShopBot** | `{"configurable": {"thread_id": "customer-1"}}` |

In a real app the `thread_id` is your customer's id, or a ticket number.

**The payoff:** after the first message you send **only the new message**. The rest is loaded for you.

In [ ]:
# ▶️ SEE IT: same graph, two customers, two separate memories.
alice = {"configurable": {"thread_id": "alice"}}
bob   = {"configurable": {"thread_id": "bob"}}

shopbot.invoke({"messages": [HumanMessage("Hi, I'm Alice.")], "attempts": 0}, alice)
# Second message: we send ONLY the new one - the checkpointer remembers the rest.
out = shopbot.invoke({"messages": [HumanMessage("What's my name?")], "attempts": 0}, alice)
print("to Alice:", out["messages"][-1].content[:90])

out = shopbot.invoke({"messages": [HumanMessage("What's my name?")], "attempts": 0}, bob)
print("to Bob  :", out["messages"][-1].content[:90])

print("\nsaved messages - alice:", len(shopbot.get_state(alice).values["messages"]),
      "| bob:", len(shopbot.get_state(bob).values["messages"]))
print("Same bot. Separate memories. That's what thread_id does.")

## 2.18 Snapshot, `get_state` and `.next`

| | |
|---|---|
| 🗣️ **Plain words** | A **snapshot** is one saved moment. `get_state()` fetches the latest one. `.next` tells you which step is waiting to run. |
| 🏠 **Analogy** | Looking at your autosave: here's my inventory, and here's the quest I'm on. |
| 👉 **In ShopBot** | `shopbot.get_state(cfg).values` and `.next` |

`get_state()` is the **single most useful debugging call in LangGraph**. When something's wrong, look at what was actually saved.

**`.next` is how you know whether a run finished:**

- `()` — an empty tuple → finished
- `("wait_for_human",)` → paused, waiting at that step

In [ ]:
# ▶️ SEE IT
snap = shopbot.get_state(cfg)
print("saved keys      :", list(snap.values.keys()))
print("messages saved  :", len(snap.values["messages"]))
print("next step to run:", snap.next, "  <- empty tuple means it FINISHED")

## 2.19 Interrupt and `Command(resume=…)`

| | |
|---|---|
| 🗣️ **Plain words** | `interrupt()` **stops the graph mid-run** and hands control back to your app. `Command(resume=…)` continues from exactly that spot. |
| 🏠 **Analogy** | A cashier who needs a manager's key for a refund. Everything freezes until the manager comes over. |
| 👉 **In ShopBot** | `decision = interrupt({...})` inside `wait_for_human` |

Because the state is **saved** (that's the checkpointer again), the pause can last a millisecond or three days, and can be continued **by a different computer** — a click in an admin panel, a Slack button, an email reply.

**Two things to know:**

1. A paused run returns a special key: `__interrupt__`. Check for it rather than assuming.
2. To continue, you pass `Command(resume=…)` and **no new message**.

In [ ]:
# ▶️ SEE IT: a refund big enough to need a human.
refund_cfg = {"configurable": {"thread_id": "refund-demo"}}

paused = shopbot.invoke(
    {"messages": [HumanMessage("My monitor B-200 is 11 days late. I want a refund.")],
     "attempts": 0},
    refund_cfg,
)

if "__interrupt__" in paused:
    print("\n⏸ PAUSED. Your app would show this to a manager:")
    print("   ", paused["__interrupt__"][0].value["question"])
    print("   waiting at step:", shopbot.get_state(refund_cfg).next)
    print("   Nothing has been refunded yet. That is the whole point.")
else:
    print("\n(no pause this time - the model may not have flagged a refund)")
    print("reply:", paused.get("messages", [AIMessage("")])[-1].content[:120])

In [ ]:
# ▶️ SEE IT: the manager approves. In a real app this call happens later,
# possibly from a completely different server.
still_waiting = bool(shopbot.get_state(refund_cfg).next)

if still_waiting:
    done = shopbot.invoke(Command(resume={"approve": True}), refund_cfg)
    #                     ^ no new message needed - just the decision
    print("▶️ CONTINUED:\n")
    print(done["messages"][-1].content)
else:
    print("(nothing was paused, so there is nothing to continue)")

## 2.20 Prebuilt

| | |
|---|---|
| 🗣️ **Plain words** | Ready-made pieces so you don't build common patterns by hand. |
| 🏠 **Analogy** | Flat-pack furniture instead of cutting your own timber. |
| 👉 **Not used in ShopBot** — we built everything by hand on purpose, so you could see it |

| Prebuilt | What it saves you |
|---|---|
| `ToolNode` | running whichever function the model asked for |
| `tools_condition` | the "did it ask for a tool? then loop" decision |
| `create_agent` | a whole standard tool-using agent in one line |

Learn these **after** you're comfortable with §2.1–2.19. Knowing the hand-built version is what lets you fix things when the prebuilt one doesn't fit.

# 3. LangSmith terminology

LangSmith is a **website that records what your code did**. Fourteen terms.

Set the key below to see the real dashboard — or leave it empty; every cell still runs.

In [ ]:
LANGSMITH_KEY = ""        # paste "lsv2_pt_..." here, or leave empty to skip

if LANGSMITH_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_KEY
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "shopbot"
    print("Tracing ON -> project 'shopbot' at https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("Tracing OFF (no key). Every cell below still runs.")

TRACING_ON = bool(LANGSMITH_KEY)

Tracing OFF (no key). Every cell below still runs.


## 3.1 Tracing

| | |
|---|---|
| 🗣️ **Plain words** | Automatically recording every step your app took, with its inputs and outputs. |
| 🏠 **Analogy** | A dashcam. You don't think about it until something happens, then it's the only thing that matters. |
| 👉 **In ShopBot** | Three environment variables. **You change no code at all.** |

The single most useful thing it records: **the exact prompt that was sent to the model.** Nine times out of ten, reading the real prompt ends the mystery instantly — a blank wasn't filled in, or the history was longer than you thought.

## 3.2 Trace

| | |
|---|---|
| 🗣️ **Plain words** | The recording of **one complete run** of your app. |
| 🏠 **Analogy** | One receipt from one shopping trip, itemised. |
| 👉 **In ShopBot** | One `shopbot.invoke(...)` = one trace |

## 3.3 Run and span

| | |
|---|---|
| 🗣️ **Plain words** | One **step** inside a trace. "Run" and "span" mean the same thing here. |
| 🏠 **Analogy** | One line item on the receipt. |
| 👉 **In ShopBot** | `classify` is a run. The model call inside it is another run, nested underneath |

## 3.4 Root run, parent, child

| | |
|---|---|
| 🗣️ **Plain words** | Runs nest inside each other like folders. The outermost one is the **root**. |
| 🏠 **Analogy** | A file explorer tree. |

That's what a trace of our ShopBot run looks like:

```
shopbot                                     3.4s   $0.0003     ← root run
├── classify                                0.9s
│   └── ChatOpenAI                          0.8s   142 tokens  ← child run
├── look_up_order                           0.0s               ← our plain Python
├── draft                                   1.2s
│   └── ChatOpenAI                          1.1s   198 tokens
├── check                                   0.9s
│   └── ChatOpenAI                          0.9s   210 tokens
└── send                                    0.0s
```

You can immediately see: which step was slow, how many tokens each cost, and — by clicking `ChatOpenAI` — the exact text that was sent.

In [ ]:
# ▶️ SEE IT: run ShopBot again. With tracing on, this creates one trace with
# several nested runs. The code is IDENTICAL either way - that's the point.
out = shopbot.invoke({"messages": [HumanMessage("Where is A-100?")], "attempts": 0},
                     {"configurable": {"thread_id": "trace-demo"}})
print("\nreply:", out["messages"][-1].content[:120])

if TRACING_ON:
    print("\n👉 Go to the website. Open the newest trace and expand the tree above.")
else:
    print("\n(Nothing was uploaded, since tracing is off. Notice we didn't change any code.)")

   classify -> track (order A-100)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done

reply: Hello! Your order A-100, the desk lamp, was delivered 3 days ago. The total price was $42.00. If you have any issues or 

(Nothing was uploaded, since tracing is off. Notice we didn't change any code.)


## 3.5 Project

| | |
|---|---|
| 🗣️ **Plain words** | A folder for traces. |
| 🏠 **Analogy** | A folder in your email for one client. |
| 👉 **In ShopBot** | `LANGSMITH_PROJECT = "shopbot"` — created automatically the first time |

Common habit: one project per app **per environment** — `shopbot-dev`, `shopbot-prod` — so your experiments don't pollute your real traffic.

## 3.6 `run_name`, tags, metadata

| | |
|---|---|
| 🗣️ **Plain words** | Labels you attach so you can find runs later. |
| 🏠 **Analogy** | Writing on a box before you put it in the attic. Without a label, good luck. |
| 👉 **In ShopBot** | the `config={"run_name": ..., "tags": [...], "metadata": {...}}` argument |

| Field | Use it for |
|---|---|
| `run_name` | the name in the list — otherwise everything is called `RunnableSequence` |
| `tags` | filtering: `"v1"` vs `"v2"`, `"refund"` vs `"tracking"` |
| `metadata` | any extra facts: which customer, which plan, which setting |

Costs nothing when tracing is off, so leave it in your code.

In [ ]:
# ▶️ SEE IT: the same call, labelled.
out = shopbot.invoke(
    {"messages": [HumanMessage("Where is A-100?")], "attempts": 0},
    {
        "configurable": {"thread_id": "labelled-demo"},
        "run_name": "shopbot_tracking",                     # the name you'll see
        "tags": ["shopbot", "tracking", "v1"],              # filter by these
        "metadata": {"customer_id": "c-42", "plan": "free"},# search by these
    },
)
print("reply:", out["messages"][-1].content[:100])
print("\nNow you can ask the website: 'show me all v1 tracking runs for customer c-42'.")

   classify -> track (order A-100)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done
reply: Hello! Your order A-100, the desk lamp, was delivered 3 days ago. The total price was $42.00. If you

Now you can ask the website: 'show me all v1 tracking runs for customer c-42'.


## 3.7 `@traceable` and `run_type`

| | |
|---|---|
| 🗣️ **Plain words** | A one-line decorator that makes **your own** Python show up in the trace. |
| 🏠 **Analogy** | Clipping a microphone onto someone who wasn't already miked up. |
| 👉 **In ShopBot** | our `look_up()` function is plain Python — LangSmith can't see it until we decorate it |

`run_type` just tells the website how to draw it: `"chain"`, `"llm"`, `"retriever"`, `"tool"`, `"prompt"`.

In [ ]:
# ▶️ SEE IT
@traceable(run_type="tool", name="order_database_lookup")
def look_up_traced(order_id: str) -> str:
    """Exactly the same function - now visible in the trace as its own step."""
    return look_up(order_id)

print(look_up_traced("B-200"))
print("\nWith tracing on, this appears as a step called 'order_database_lookup',")
print("sitting alongside the model calls, with its own timing.")
print("The decorator does nothing when tracing is off, so it's safe to leave in.")

Order B-200: monitor, status DELAYED, ordered 11 days ago, price $310.00.

With tracing on, this appears as a step called 'order_database_lookup',
sitting alongside the model calls, with its own timing.
The decorator does nothing when tracing is off, so it's safe to leave in.


## 3.8 Latency, tokens, cost

| | |
|---|---|
| 🗣️ **Plain words** | How long it took · how much text went in and out · what that cost. |
| 🏠 **Analogy** | The three numbers on a taxi meter. |
| 👉 **In ShopBot** | recorded automatically for each model call |

A **token** is roughly 4 characters (about ¾ of a word). You pay per token, both directions.

You don't need LangSmith to see these — every reply carries them.

In [ ]:
# ▶️ SEE IT: without any website at all.
reply = llm.invoke("Say hello in five words.")
print("tokens:", reply.usage_metadata)
print("\ninput_tokens  = everything you SENT (rules + history + question)")
print("output_tokens = what the model WROTE (usually costs several times more)")
print("\nLog this per request and you can chart cost per customer.")

tokens: {'input_tokens': 13, 'output_tokens': 7, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

input_tokens  = everything you SENT (rules + history + question)
output_tokens = what the model WROTE (usually costs several times more)

Log this per request and you can chart cost per customer.


## 3.9 Feedback

| | |
|---|---|
| 🗣️ **Plain words** | A score attached to a specific run — usually a thumbs up or down from the user. |
| 🏠 **Analogy** | The "was this helpful?" buttons at the bottom of a help article. |
| 👉 **In ShopBot** | you'd add buttons to the reply, then send the score with the run's id |

📌 **The habit that matters:** every thumbs-down becomes a test case (§3.11). That loop — real failure → test → fix → prove it — is the difference between a demo and a product.

In [ ]:
# ▶️ SEE IT: give the run an id so feedback can be attached to it later.
import uuid
run_id = str(uuid.uuid4())         # a unique label for this one run

out = shopbot.invoke(
    {"messages": [HumanMessage("Where is B-200?")], "attempts": 0},
    {"configurable": {"thread_id": "feedback-demo"}, "run_id": run_id},
)
print("reply:", out["messages"][-1].content[:90])
print("\nrun_id:", run_id[:8], "...")

if TRACING_ON:
    from langsmith import Client
    Client().create_feedback(run_id=run_id, key="user_score", score=1,
                             comment="clear and correct")
    print("thumbs-up uploaded and attached to that exact run")
else:
    print("(without a key you'd store this in your own table - the point is to CAPTURE it)")

   classify -> track (order B-200)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done
reply: Dear Customer,  

Your order B-200 for the monitor, priced at $310.00, is currently delaye

run_id: 2be32aeb ...
(without a key you'd store this in your own table - the point is to CAPTURE it)


## 3.10 Offline vs online evaluation

| | |
|---|---|
| 🗣️ **Plain words** | **Offline** = testing before release, on questions you chose. **Online** = watching real users after release. |
| 🏠 **Analogy** | Offline is a driving test. Online is your insurance company tracking how you actually drive. |

You need both. Offline proves a change is safe; online catches what you never thought to test.

## 3.11 Dataset and Example

| | |
|---|---|
| 🗣️ **Plain words** | A **dataset** is a saved list of test questions. Each one is an **example**. |
| 🏠 **Analogy** | A past exam paper with the answers in the back. |
| 👉 **In ShopBot** | the `TESTS` list in the next cell |

An example has two parts:

| Part | Meaning |
|---|---|
| **inputs** | what your app receives ("Where is B-200?") |
| **reference output** (or "expected output", "ground truth") | the right answer ("DELAYED") |

⚠️ Always include questions your app **can't** answer, so you test that it says "I don't know" instead of inventing something.

In [ ]:
# ▶️ SEE IT: a dataset is honestly just a list of dicts.
TESTS = [
    {"message": "Where is my order A-100?",        "expect_intent": "track",  "must_contain": "DELIVERED"},
    {"message": "What's happening with B-200?",    "expect_intent": "track",  "must_contain": "DELAYED"},
    {"message": "I want my money back for B-200",  "expect_intent": "refund", "must_contain": None},
    {"message": "hello there",                     "expect_intent": "chat",   "must_contain": None},
    {"message": "thanks!",                         "expect_intent": "chat",   "must_contain": None},
]

print(f"{len(TESTS)} examples")
print("\none example:")
print("  inputs           :", TESTS[1]["message"])
print("  reference outputs:", {"intent": TESTS[1]["expect_intent"],
                               "must_contain": TESTS[1]["must_contain"]})

5 examples

one example:
  inputs           : What's happening with B-200?
  reference outputs: {'intent': 'track', 'must_contain': 'DELAYED'}


## 3.12 Evaluator, score, key

| | |
|---|---|
| 🗣️ **Plain words** | An **evaluator** is a function that marks one answer. The **score** is the mark. The **key** is the name of the thing being measured. |
| 🏠 **Analogy** | A teacher with a marking scheme. "Spelling: 1. Grammar: 0." |
| 👉 **In ShopBot** | the two functions in the next cell |

Two families:

| Type | Example | Verdict |
|---|---|---|
| **Heuristic** (plain code) | "does the reply contain the word DELAYED?" | free, instant, never flaky — **use this whenever you can** |
| **LLM-as-judge** | "is this reply polite and accurate?" | for fuzzy things code can't check; costs money |

`score: None` means **"this measure doesn't apply to this example"** — better than scoring it zero, which would drag your average down unfairly.

In [ ]:
# ▶️ SEE IT: two heuristic evaluators. No AI, no cost, no flakiness.
def eval_routing(actual_intent: str, expected: str) -> dict:
    """Did it go down the right path?"""
    return {"key": "routing", "score": 1.0 if actual_intent == expected else 0.0}

def eval_mentions_fact(answer: str, must_contain) -> dict:
    """Did the reply include the fact we know it should?"""
    if must_contain is None:
        return {"key": "mentions_fact", "score": None}    # not applicable here
    return {"key": "mentions_fact",
            "score": 1.0 if must_contain.lower() in answer.lower() else 0.0}

print(eval_routing("track", "track"))
print(eval_routing("chat", "track"))
print(eval_mentions_fact("Your order is DELAYED, sorry!", "DELAYED"))
print(eval_mentions_fact("Hello!", None), "  <- None = doesn't apply")

{'key': 'routing', 'score': 1.0}
{'key': 'routing', 'score': 0.0}
{'key': 'mentions_fact', 'score': 1.0}
{'key': 'mentions_fact', 'score': None}   <- None = doesn't apply


## 3.13 Experiment

| | |
|---|---|
| 🗣️ **Plain words** | One complete marking session: run your app over the whole dataset and record the scores. |
| 🏠 **Analogy** | Sitting the whole past paper and totting up your marks. |
| 👉 **In ShopBot** | the `run_experiment()` function below |

The reason experiments exist, and the only method you need:

> **Change exactly one thing. Run the same dataset again. Keep the change only if the number went up.**

In [ ]:
# ▶️ SEE IT: a whole experiment, in plain Python. No account needed.
def run_experiment(label: str = "v1") -> dict:
    routing_scores, fact_scores = [], []

    print(f"{'ROUTE':<8}{'FACT':<8}MESSAGE")
    for i, t in enumerate(TESTS):
        # A fresh thread per test, so one answer can't leak into the next.
        out = shopbot.invoke(
            {"messages": [HumanMessage(t["message"])], "attempts": 0},
            {"configurable": {"thread_id": f"exp-{label}-{i}"}, "tags": [label]},
        )
        answer = out.get("messages", [AIMessage("")])[-1].content

        r = eval_routing(out.get("intent", "?"), t["expect_intent"])
        f = eval_mentions_fact(answer, t["must_contain"])

        routing_scores.append(r["score"])
        if f["score"] is not None:
            fact_scores.append(f["score"])

        mark = lambda s: "-" if s is None else ("PASS" if s >= 1 else "FAIL")
        print(f"{mark(r['score']):<8}{mark(f['score']):<8}{t['message'][:40]}")

    scores = {"routing": round(sum(routing_scores) / len(routing_scores), 2)}
    if fact_scores:
        scores["mentions_fact"] = round(sum(fact_scores) / len(fact_scores), 2)

    print(f"\nSCORECARD [{label}]: {scores}")
    return scores

v1 = run_experiment("v1")

ROUTE   FACT    MESSAGE
   classify -> track (order A-100)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done
PASS    PASS    Where is my order A-100?
   classify -> track (order B-200)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done
PASS    PASS    What's happening with B-200?
   classify -> refund (order B-200)
   look_up_order -> done
   draft -> try 1
   check -> ok
   wait_for_human -> PAUSED ($310.00)
PASS    -       I want my money back for B-200
   classify -> chat (order None)
   chat -> done
PASS    -       hello there
   classify -> chat (order None)
   chat -> done
PASS    -       thanks!

SCORECARD [v1]: {'routing': 1.0, 'mentions_fact': 1.0}


## 3.14 Annotation queue · monitoring · playground

Three more words you'll see. You don't need them today, but here's what they mean.

| Term | 🗣️ In plain words | 🏠 Analogy |
|---|---|---|
| **Annotation queue** | a to-do list of tricky runs for a human expert to mark by hand | a doctor reviewing X-rays the software wasn't sure about |
| **Monitoring / dashboard** | charts over time: cost, speed, error rate, thumbs-downs | the gauges on a car dashboard |
| **Playground** | a web page for editing a prompt and re-running it without touching your code | a sandbox for trying things |
| **Prompt hub** | saved, version-numbered prompts your team shares | Google Docs version history, for prompts |

In [ ]:
# ▶️ SEE IT: what monitoring measures, done manually.
# These four numbers are what an operations person will ask you for.
start = time.time()
out = shopbot.invoke({"messages": [HumanMessage("Where is A-100?")], "attempts": 0},
                     {"configurable": {"thread_id": "monitor-demo"}})
elapsed = time.time() - start

print("latency        :", round(elapsed, 2), "seconds")
print("path taken     :", out.get("intent"))
print("draft attempts :", out.get("attempts"), "(more than 1 means the reviewer rejected one)")
print("refusal / error: none")
print("\nA dashboard is just these numbers, collected for every request and plotted.")

   classify -> track (order A-100)
   look_up_order -> done
   draft -> try 1
   check -> ok
   send -> done
latency        : 3.48 seconds
path taken     : track
draft attempts : 1 (more than 1 means the reviewer rejected one)
refusal / error: none

A dashboard is just these numbers, collected for every request and plotted.


# 4. Cheat sheets

## LangGraph in one table

| Term | 🗣️ Plain words | 🏠 Analogy | Code |
|---|---|---|---|
| **Graph** | your app as a flowchart | an Underground map | `StateGraph(...)` |
| **State** | a dictionary passed between steps | a hospital clipboard | `class S(TypedDict)` |
| **Node** | one step (a normal function) | one station | `add_node("name", fn)` |
| **Partial update** | return only what changed | writing one new line on the clipboard | `return {"key": value}` |
| **Edge** | "then do this next" | the track between stations | `add_edge("a", "b")` |
| **START / END** | begin here / stop here | "you are here" / exit sign | `add_edge(START, "a")` |
| **Conditional edge** | an arrow that splits; a function picks | a railway switch | `add_conditional_edges(...)` |
| **Router** | the picking function | the signal operator | `def choose(state) -> str` |
| **Cycle** | an arrow going backwards | taste, salt, taste again | `add_edge("check", "draft")` |
| **Recursion limit** | cap on total steps | a kettle that switches off | your own counter + the default 25 |
| **Reducer** | how new values combine with old | replace vs append to a list | `Annotated[list, fn]` |
| **`add_messages`** | the ready-made "append" reducer | stapling a page to a file | `Annotated[list, add_messages]` |
| **Compile** | check the plan, make it runnable | the surveyor approving drawings | `builder.compile()` |
| **Invoke** | run it once | pressing start | `graph.invoke(input, config)` |
| **Config** | settings for this run | the programme dial | `{"configurable": {...}}` |
| **Stream** | updates as they happen | a pizza tracker | `graph.stream(..., stream_mode=)` |
| **Checkpointer** | saves state after every step | a game autosave | `compile(checkpointer=...)` |
| **Thread / `thread_id`** | one conversation and its name | a WhatsApp chat | `{"thread_id": "alice"}` |
| **Snapshot / `get_state`** | look at what was saved | opening your autosave | `graph.get_state(config)` |
| **`.next`** | which step is waiting | the quest you're on | `get_state(cfg).next` |
| **Interrupt** | stop and wait for a person | needing a manager's key | `interrupt({...})` |
| **`Command(resume=)`** | carry on from the pause | the manager turns the key | `invoke(Command(resume=x), cfg)` |
| **Prebuilt** | ready-made common pieces | flat-pack furniture | `ToolNode`, `create_agent` |

## LangSmith in one table

| Term | 🗣️ Plain words | 🏠 Analogy | Code |
|---|---|---|---|
| **Tracing** | auto-recording every step | a dashcam | 3 environment variables |
| **Trace** | the recording of one run | one itemised receipt | one `.invoke()` |
| **Run / span** | one step inside a trace | one line on the receipt | automatic |
| **Root / parent / child** | steps nested inside steps | a folder tree | automatic |
| **Project** | a folder of traces | an email folder | `LANGSMITH_PROJECT` |
| **`run_name`** | the name in the list | writing on the box | `config={"run_name": ...}` |
| **Tags** | filterable labels | coloured stickers | `config={"tags": [...]}` |
| **Metadata** | extra facts about the run | the contents list on the box | `config={"metadata": {...}}` |
| **`@traceable`** | makes your own code visible | clipping on a microphone | `@traceable(name="x")` |
| **`run_type`** | how to draw the step | which icon to use | `run_type="tool"` |
| **Latency / tokens / cost** | time, text volume, money | a taxi meter | `reply.usage_metadata` |
| **Feedback** | a score on one run | "was this helpful?" | `create_feedback(run_id=...)` |
| **Dataset** | a saved list of test questions | a past exam paper | a list of dicts |
| **Example** | one question in it | one exam question | `{"inputs":..., "outputs":...}` |
| **Reference output** | the right answer | the answers in the back | `expect_intent` |
| **Evaluator** | a function that marks an answer | a marking scheme | returns `{"key","score"}` |
| **Score / key** | the mark / what's measured | "spelling: 1" | `{"key":"routing","score":1.0}` |
| **Heuristic vs judge** | code check vs a second model | a spell-checker vs a human marker | free vs costs money |
| **Experiment** | one full marking session | sitting the whole paper | run over the dataset |
| **Offline / online** | test before / watch after | driving test / insurance tracker | dataset / real traffic |
| **Annotation queue** | tricky runs for a human to mark | a doctor's second opinion | web UI |
| **Monitoring** | charts over time | car dashboard gauges | web UI |
| **Playground / hub** | edit prompts on the web | a sandbox / version history | web UI |

## The five sentences that matter most

1. **A graph is boxes and arrows.** State is the clipboard passed between them.
2. **A node is just a function.** No magic, which is why it's easy to test.
3. **A checkpointer + `thread_id` gives you memory, crash recovery, pausing and separate users** — from one argument.
4. **Every loop needs your own counter.** The built-in limit is a safety net, not a plan.
5. **A trace shows you the exact prompt that was sent.** That one fact solves most mysteries.

## Words you'll hear but don't need yet

| Word | It means | Learn it when |
|---|---|---|
| **Tool / tool calling** | the model asks your code to run a function | you want the model to look things up itself |
| **Agent** | a model that chooses its own next step, in a loop | after you're solid on graphs |
| **ReAct** | the think → act → observe → repeat pattern | same time as agents |
| **Structured output** | forcing a fixed data shape back from the model | straight away, honestly — it's the most useful one |
| **RAG** | answering from your own documents | when you have documents |
| **Embedding / vector store** | the machinery underneath RAG | same time as RAG |
| **Super-step** | one full "tick" of the graph, which may run several nodes at once | when you need parallel steps |
| **Durability** | how often the checkpointer saves | when you tune performance |

🧪 **Final exercise:** add a `"complaint"` intent to ShopBot. You'll need one new value in the `Intent` `Literal`, one new node, one more name in a conditional edge's list, and one more line in `after_classify`. Then add a test for it to `TESTS` and re-run `run_experiment("v2")` — and you'll have used almost every term in both tables.